# JONICA26 — Conversión del modelo `.pt` → `.hef`

Convierte el YOLOv8n de detección de **PARE** y **semáforo verde** al formato
que corre en el acelerador **Hailo-8** del AI HAT+ de 26 TOPS.

### Por qué en Colab y no en tu máquina

El Hailo Dataflow Compiler (DFC) es lo único que produce archivos `.hef`, y
sólo corre en **Linux x86_64 con instrucciones AVX**. No funciona en macOS,
ni en Apple Silicon, ni en la propia Raspberry Pi. Colab es x86 y gratis, así
que es el camino más corto.

### Antes de empezar necesitás

1. **El wheel del DFC, versión 3.33.x.** Se baja de la
   [Hailo Developer Zone](https://hailo.ai/developer-zone/) (cuenta gratuita).
   Buscá *Dataflow Compiler* y bajá el `.whl` para **Python 3.10**.
   **Este paso es manual y no se puede automatizar**: está detrás del login.

   > ⚠️ **Cuidado con la versión.** La del compilador tiene que coincidir con
   > la del runtime instalado en la Raspberry. Verificalo con
   > `hailortcli fw-control identify` en la Pi:
   >
   > | HailoRT en la Pi | DFC a bajar |
   > |---|---|
   > | 4.23.x | **3.33.x** |
   > | 4.22.x | 3.32.x |
   > | 4.21.x | 3.31.x |
   >
   > **No bajes la línea 5.x**, aunque sea la más nueva y la que aparece
   > primero: esa es para Hailo-10H y Hailo-15. Un `.hef` compilado con DFC
   > 5.x no carga en un Hailo-8.
   >
   > Si en la Developer Zone sólo ves la 5.x, buscá en la sección de
   > archivo o en la del producto Hailo-8 / AI Software Suite.
2. **El modelo** `yolov8npareysemaforo.pt` (está en `model/` del repo).
3. **Tu API key de Roboflow**, para bajar el dataset `pareysemaforo-3` que se
   usa como set de calibración.

### Importante: usar GPU no sirve
El DFC compila en CPU. En *Entorno de ejecución → Cambiar tipo de entorno*
dejá **CPU**, y si podés elegir, un entorno con **alta RAM**.

---

## Paso 0 — Verificar que el entorno sirve

Si esta celda falla, el resto no va a funcionar.

In [ ]:
import platform, subprocess, sys

arch = platform.machine()
print(f"Arquitectura: {arch}")
assert arch == "x86_64", f"El DFC necesita x86_64, este entorno es {arch}"

flags = subprocess.run(["grep", "-m1", "flags", "/proc/cpuinfo"],
                       capture_output=True, text=True).stdout
print("AVX:   ", "avx" in flags)
print("AVX2:  ", "avx2" in flags)
assert "avx" in flags, "Esta CPU no tiene AVX y el DFC no va a poder correr"

ram_gb = int(subprocess.run(["grep", "MemTotal", "/proc/meminfo"],
             capture_output=True, text=True).stdout.split()[1]) / 1024 / 1024
print(f"RAM:    {ram_gb:.1f} GB")
if ram_gb < 12:
    print("AVISO: con menos de 12 GB la compilacion puede quedarse sin memoria.")

print(f"\nUbuntu: {platform.version()}")
print("Entorno OK para compilar.")

## Paso 1 — Entorno Python 3.10

El DFC pide **Python 3.8–3.10**, y Colab viene con una versión más nueva. Por
eso armamos un venv aparte con 3.10 y ahí adentro instalamos todo. El resto
del notebook llama a ese intérprete con la variable `PY`.

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq python3.10 python3.10-venv python3.10-dev build-essential > /dev/null
!python3.10 -m venv /content/hailo_env
PY = "/content/hailo_env/bin/python"
PIP = "/content/hailo_env/bin/pip"
!{PY} --version
!{PIP} install -q --upgrade pip setuptools wheel

### Subir el wheel del DFC

Dos opciones: subirlo a mano acá, o dejarlo en tu Google Drive y montarlo
(más cómodo si vas a repetir la conversión, porque el wheel pesa bastante).

In [ ]:
# --- Opcion A: subir el archivo a mano ---
from google.colab import files
print("Elegi el archivo hailo_dataflow_compiler-*.whl que bajaste de la Developer Zone")
subidos = files.upload()
WHEEL = "/content/" + next(iter(subidos))

# --- Opcion B: desde Google Drive (comentar la Opcion A y descomentar esto) ---
# from google.colab import drive; drive.mount('/content/drive')
# WHEEL = "/content/drive/MyDrive/hailo/hailo_dataflow_compiler-3.33.0-py3-none-linux_x86_64.whl"

print("Wheel:", WHEEL)

In [ ]:
# La instalacion del DFC tarda varios minutos: arrastra TensorFlow y ONNX.
!{PIP} install {WHEEL}
!{PIP} install -q git+https://github.com/hailo-ai/hailo_model_zoo.git

# Verificacion
!{PY} -c "from hailo_sdk_client import ClientRunner; print('DFC importado OK')"
!/content/hailo_env/bin/hailomz --version

## Paso 2 — Exportar el `.pt` a ONNX

El DFC no lee PyTorch: el paso intermedio obligatorio es ONNX.

Dos detalles que importan:

- **`opset=11`** es el que mejor digiere el parser de Hailo.
- **Sin NMS en el ONNX.** El NMS se agrega después, compilado *dentro* del
  `.hef`, para que corra en el chip y no en la CPU de la Raspberry. Por eso
  acá no se pasa `nms=True`.

In [ ]:
from google.colab import files
print("Subi yolov8npareysemaforo.pt (esta en model/ del repo)")
subidos = files.upload()
PT = "/content/" + next(iter(subidos))

# Ultralytics va en el Python de Colab, no en el venv del DFC: son entornos
# distintos y no necesitan verse.
!pip install -q ultralytics==8.2.103

from ultralytics import YOLO
modelo = YOLO(PT)
print("Clases del modelo:", modelo.names)
assert len(modelo.names) == 2, f"Se esperaban 2 clases, hay {len(modelo.names)}"

ONNX = modelo.export(format="onnx", opset=11, imgsz=640, simplify=True)
print("ONNX generado:", ONNX)

## Paso 3 — Dataset de calibración

Para cuantizar a INT8 el compilador necesita ver **imágenes reales** y medir
qué rango de valores toma cada capa. La documentación de Hailo pide más de
1024 imágenes.

Esto es lo que más influye en que el modelo siga detectando bien después de
convertir: las imágenes tienen que parecerse a lo que va a ver el robot en la
pista. Por eso usamos el mismo dataset con el que se entrenó.

In [ ]:
!pip install -q roboflow

from roboflow import Roboflow
import getpass

api_key = getpass.getpass("API key de Roboflow: ")
rf = Roboflow(api_key=api_key)

# Ajustar workspace/proyecto/version si no coinciden con tu cuenta.
proyecto = rf.workspace().project("pareysemaforo")
dataset = proyecto.version(3).download("yolov8", location="/content/dataset")
print("Dataset en:", dataset.location)

In [ ]:
from pathlib import Path
from PIL import Image
import shutil, random

CALIB = Path("/content/calib")
if CALIB.exists():
    shutil.rmtree(CALIB)
CALIB.mkdir(parents=True)

origen = list(Path("/content/dataset/train/images").glob("*.jpg"))
origen += list(Path("/content/dataset/train/images").glob("*.png"))
# Si no alcanzan, completamos con las de validacion.
if len(origen) < 1024:
    origen += list(Path("/content/dataset/valid/images").glob("*.jpg"))

random.seed(42)
random.shuffle(origen)
seleccion = origen[:1024]

for i, ruta in enumerate(seleccion):
    Image.open(ruta).convert("RGB").resize((640, 640)).save(CALIB / f"calib_{i:05d}.jpg")

print(f"Imagenes disponibles: {len(origen)}")
print(f"Imagenes de calibracion preparadas: {len(list(CALIB.glob('*.jpg')))}")
if len(seleccion) < 1024:
    print("\nAVISO: menos de 1024 imagenes. Se puede compilar igual, pero")
    print("conviene revisar bien el resultado del Paso 5.")

## Paso 4 — Compilar a `.hef`

Usamos el **Hailo Model Zoo** (`hailomz`) en vez del DFC crudo, porque ya trae
resueltos los dos puntos donde más se traba la gente con YOLOv8:

- los **end nodes** (los 6 nodos donde hay que cortar el modelo antes del
  decode), y
- la configuración del **NMS** que se compila dentro del `.hef`.

⚠️ **`--hw-arch hailo8`** porque el AI HAT+ es el de 26 TOPS. Si fuera el de
13 TOPS iría `hailo8l`, y un `.hef` **no es intercambiable** entre los dos.

Esta celda tarda bastante: entre 15 y 40 minutos según el entorno.

In [ ]:
# Verificar que hailo8 este entre los targets soportados por esta version.
!/content/hailo_env/bin/hailomz compile --help 2>&1 | grep -A3 "hw-arch" | head -10

In [ ]:
%%time
!cd /content && /content/hailo_env/bin/hailomz compile yolov8n \
    --ckpt {ONNX} \
    --hw-arch hailo8 \
    --calib-path /content/calib \
    --classes 2 \
    --performance

In [ ]:
from pathlib import Path
import shutil

hefs = list(Path("/content").rglob("*.hef"))
print("Archivos .hef encontrados:", [str(h) for h in hefs])
assert hefs, "No se genero ningun .hef. Revisar los errores de la celda anterior."

HEF = max(hefs, key=lambda p: p.stat().st_mtime)
DESTINO = "/content/yolov8npareysemaforo.hef"
shutil.copy(HEF, DESTINO)
print(f"\n{DESTINO}  ({Path(DESTINO).stat().st_size / 1e6:.1f} MB)")

## Paso 5 — Verificar la cuantización ⚠️ **no saltear**

Acá está el riesgo real de todo el proceso: un modelo puede **compilar
perfecto y detectar mucho peor**. La cuantización a INT8 no falla con un
error, degrada en silencio. Es un problema reportado seguido con modelos
YOLOv8 custom.

Hacemos dos cosas:

1. Generamos `referencia_pt.json` con lo que detecta el modelo **original**.
2. Comparamos contra el modelo **ya cuantizado**, usando el emulador del DFC
   — o sea, medimos la pérdida sin necesidad de tener la Hailo a mano.

Después, en la Raspberry, `tools/verificar_hef.py` repite la comparación
contra el hardware real.

In [ ]:
import json, random
from pathlib import Path

# Set de prueba: imagenes de validacion, que NO se usaron para calibrar.
prueba = sorted(Path("/content/dataset/valid/images").glob("*.jpg"))[:50]
print(f"Imagenes de prueba: {len(prueba)}")

MUESTRAS = Path("/content/muestras")
MUESTRAS.mkdir(exist_ok=True)

referencia = {}
for ruta in prueba:
    r = modelo.predict(str(ruta), imgsz=640, conf=0.25, verbose=False)[0]
    dets = []
    for caja in r.boxes:
        x1, y1, x2, y2 = caja.xyxyn[0].tolist()   # normalizado 0-1
        dets.append({
            "nombre": modelo.names[int(caja.cls[0])],
            "confianza": float(caja.conf[0]),
            "bbox": [x1, y1, x2, y2],
        })
    referencia[ruta.name] = dets
    Image.open(ruta).convert("RGB").resize((640, 640)).save(MUESTRAS / ruta.name)

Path("/content/referencia_pt.json").write_text(json.dumps(referencia, indent=2))

total = sum(len(v) for v in referencia.values())
print(f"Detecciones del .pt original: {total}")
print("Guardado en referencia_pt.json")

In [ ]:
# Comparacion contra el modelo cuantizado, en el emulador del DFC.
import json
from pathlib import Path

script = '''
import json, sys
import numpy as np
from pathlib import Path
from hailo_sdk_client import ClientRunner, InferenceContext
from PIL import Image

har = sorted(Path("/content").rglob("*quantized*.har"))
if not har:
    har = sorted(Path("/content").rglob("*.har"))
if not har:
    print("NO_HAR"); sys.exit(1)
runner = ClientRunner(har=str(har[-1]))
print("HAR:", har[-1])

muestras = sorted(Path("/content/muestras").glob("*.jpg"))
lote = np.stack([
    np.asarray(Image.open(p).convert("RGB").resize((640, 640)), dtype=np.uint8)
    for p in muestras
])

with runner.infer_context(InferenceContext.SDK_QUANTIZED) as ctx:
    salida = runner.infer(ctx, lote)

np.save("/content/salida_cuantizada.npy", np.array(salida, dtype=object),
        allow_pickle=True)
json.dump([p.name for p in muestras], open("/content/muestras.json", "w"))
print("OK: inferencia cuantizada completada sobre", len(muestras), "imagenes")
'''
Path("/content/emular.py").write_text(script)
!{PY} /content/emular.py

In [ ]:
# Medir cuanto se perdio.
import json
import numpy as np

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    if inter <= 0: return 0.0
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

ETIQUETAS = ["PARE", "semaforoverde"]
CONF, IOU_MIN = 0.25, 0.5

referencia = json.load(open("/content/referencia_pt.json"))
nombres = json.load(open("/content/muestras.json"))
salida = np.load("/content/salida_cuantizada.npy", allow_pickle=True)

total_q = total_ref = coinc = 0
for i, nombre in enumerate(nombres):
    dets = []
    for ci, cajas in enumerate(salida[i]):
        for fila in (cajas if cajas is not None else []):
            y1, x1, y2, x2, sc = [float(v) for v in fila[:5]]
            if sc >= CONF:
                dets.append((ETIQUETAS[ci], sc, (x1, y1, x2, y2)))
    total_q += len(dets)

    esperadas = [r for r in referencia.get(nombre, []) if r["confianza"] >= CONF]
    total_ref += len(esperadas)
    usadas = set()
    for esp in esperadas:
        mejor, mejor_iou = None, 0.0
        for j, (cls, _, bb) in enumerate(dets):
            if j in usadas or cls != esp["nombre"]: continue
            v = iou(bb, esp["bbox"])
            if v > mejor_iou: mejor, mejor_iou = j, v
        if mejor is not None and mejor_iou >= IOU_MIN:
            usadas.add(mejor); coinc += 1

recall = coinc / total_ref if total_ref else 0
precision = coinc / total_q if total_q else 0
print(f"Detecciones .pt original : {total_ref}")
print(f"Detecciones cuantizado   : {total_q}")
print(f"Coincidencias            : {coinc}")
print(f"\nRecall    {recall:.1%}   (cuanto del modelo original conserva)")
print(f"Precision {precision:.1%}   (cuanto inventa de mas)")
print()
if recall >= 0.90:
    print("OK. La cuantizacion conservo el modelo, se puede bajar el .hef.")
elif recall >= 0.75:
    print("ATENCION: se perdio algo de precision. Probar en pista con cuidado.")
else:
    print("MAL: demasiada perdida. Recompilar con mas imagenes de calibracion,")
    print("o revisar que el set de calibracion represente bien la pista real.")

## Paso 6 — Bajar los archivos

Bajá los dos y ponelos en el repo:

| Archivo | Va en |
|---|---|
| `yolov8npareysemaforo.hef` | `model/` |
| `referencia_pt.json` | guardalo junto a las imágenes de prueba |

Después, en la Raspberry Pi, **primero este chequeo de 5 segundos**:

```bash
hailortcli parse-hef model/yolov8npareysemaforo.hef
```

Confirma que el runtime puede leer el archivo. Si el DFC con el que compilaste
no coincide con la versión de HailoRT de la placa, falla acá con
`Unsupported hef version` — mejor enterarse ahora que en la pista.

Recién después, la verificación de precisión:

```bash
python3 tools/verificar_hef.py --imagenes docs/muestras/ --referencia referencia_pt.json
```

Eso repite la comparación contra el hardware real. Si da bien, el robot está
listo para la pista.

In [ ]:
from google.colab import files
files.download("/content/yolov8npareysemaforo.hef")
files.download("/content/referencia_pt.json")